# Automatic Differentiation (Gradient Calculation) of Tensors in PyTorch

In [1]:
import torch

## 1. Simple function differentiation

$\frac{d}{dx}x^2 = 2x$

In [6]:
x = torch.tensor(3.0, requires_grad=True)
x

tensor(3., requires_grad=True)

In [3]:
y = x**2
y

tensor(9., grad_fn=<PowBackward0>)

In [4]:
y.backward()

In [5]:
x.grad

tensor(6.)

## 2. Nested function differentiation

$\frac{dz}{dx}$ where $z = sin(y)$ and, $y = x^2$

In [35]:
x = torch.tensor(3.0, requires_grad=True)
x

tensor(3., requires_grad=True)

In [36]:
y = x**2
y

tensor(9., grad_fn=<PowBackward0>)

In [37]:
z = torch.sin(y)
z

tensor(0.4121, grad_fn=<SinBackward0>)

In [38]:
z.backward()

In [39]:
x.grad

tensor(-5.4668)

## 3. Manual Simulation of a Complete Epoch for a Single Perceptron/Neuron

In [40]:
# Inputs
x = torch.tensor(6.7) # Input feature
y = torch.tensor(0.0) # True label (binary)

w = torch.tensor(1.0) # Weight
b = torch.tensor(0.0) # Bias

In [41]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8 # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [42]:
# Forward pass
z = w*x + b # Weighted sum (linear part)
y_pred = torch.sigmoid(z) # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [43]:
# Derivatives (Backpropagation)

# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x # dz/dw = x
dz_db = 1 # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [44]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


## 4. Simulation of a Complete Epoch for a Single Perceptron/Neuron using PyTorch

In [62]:
# Inputs
x = torch.tensor(6.7) # Input feature
y = torch.tensor(0.0) # True label (binary)

In [63]:
w = torch.tensor(1.0, requires_grad=True) # Weight
b = torch.tensor(0.0, requires_grad=True) # Bias

In [64]:
# Forward pass
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [65]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [66]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [67]:
# backpropagation
loss.backward()

In [68]:
print(f"Gradient of loss w.r.t weight (dw): {w.grad}")
print(f"Gradient of loss w.r.t bias (db): {b.grad}")

Gradient of loss w.r.t weight (dw): 6.6917619705200195
Gradient of loss w.r.t bias (db): 0.9987704753875732


## 5. Gradient Calculation on Vectors

In [69]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
x

tensor([1., 2., 3.], requires_grad=True)

In [70]:
y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [71]:
y.backward()

In [72]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

## 6. Clearing Gradients

In [73]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [74]:
y  = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [75]:
y.backward(retain_graph=True)
# y.backward()

In [76]:
x.grad

tensor(4.)

In [77]:
y.backward(retain_graph=True)

In [78]:
# Repeating the backpropagation accumulates gradients
x.grad

tensor(8.)

Use `x.grad.zero_()` before each backward call to clear any previous gradients

In [79]:
x.grad.zero_()
y.backward(retain_graph=True)

In [80]:
x.grad

tensor(4.)

## 7. Disable Gradient Tracking

In [81]:
# Disable gradient tracking for inference stage
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [82]:
y = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [83]:
y.backward()

In [84]:
x.grad

tensor(4.)

### a. requires_grad_(False)

In [85]:
x.requires_grad_(False)

tensor(2.)

In [86]:
y = x**2
y

tensor(4.)

### b. detach()

In [87]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [88]:
z = x.detach()
z, x

(tensor(2.), tensor(2., requires_grad=True))

### c. torch.no_grad()

In [89]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [90]:
with torch.no_grad():
    y = x ** 2

In [91]:
y

tensor(4.)

In [92]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)